# 🛍️ Customer Segmentation — Unsupervised Learning (K-Means Clustering)

### What is Unsupervised Learning?
In **Supervised Learning** (like Linear Regression) we give the model the answers to learn from.

In **Unsupervised Learning** there are **no answers** — the model finds hidden patterns on its own.

### What is K-Means Clustering?
> K-Means groups similar customers together into **K clusters** — like sorting people into buckets based on their behaviour.

**Our Goal:** Group 300 customers by **Annual Income** and **Monthly Spending** to discover natural customer segments.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("✅ Libraries loaded!")

## Step 2 — Create the Customer Dataset
300 customers with their **Annual Income** and **Monthly Spending**.

In [ ]:
np.random.seed(42)

# Simulate 3 realistic customer groups
inc1, sp1 = np.random.normal(30000, 5000, 100), np.random.normal(2000, 400, 100)   # Budget shoppers
inc2, sp2 = np.random.normal(90000, 8000, 100), np.random.normal(8000, 800, 100)   # High earners, high spenders
inc3, sp3 = np.random.normal(85000, 7000, 100), np.random.normal(2500, 400, 100)   # High earners, low spenders (savers)

df = pd.DataFrame({
    'Annual_Income'  : np.concatenate([inc1, inc2, inc3]).round(),
    'Monthly_Spend'  : np.concatenate([sp1,  sp2,  sp3 ]).round()
})

print(f"Dataset: {len(df)} customers")

df.to_csv( 'Customer_Data.csv', index_label=False )

df.describe().round(0)

### Visualise the Raw Data (Before Clustering)
Can you spot any natural groups just by looking?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['Annual_Income'] / 1000, df['Monthly_Spend'], 
            alpha=0.5, color='steelblue', edgecolors='white', s=40)
plt.xlabel('Annual Income ($000s)')
plt.ylabel('Monthly Spending ($)')
plt.title('Customer Data — Before Clustering')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Step 3 — Scale the Data
Income is in the tens of thousands, Spending in the hundreds.

**Scaling** puts both on the same scale so one doesn't overpower the other.

> Think of it like converting miles and kilometres to the same unit before comparing.

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df)

print("Before scaling — Annual Income range: ", df['Annual_Income'].min(), "to", df['Annual_Income'].max())
print("After scaling  — Annual Income range: ", round(X_scaled[:,0].min(), 2), "to", round(X_scaled[:,0].max(), 2))

## Step 4 — Find the Right Number of Clusters (Elbow Method)

We need to decide **how many groups (K)** to split customers into.

The **Elbow Method** runs K-Means with K = 1, 2, 3... and plots the error.

> 📍 Look for where the line bends like an **elbow** — that's the ideal K.

In [ ]:
inertia = []
K_range = range(1, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, inertia, marker='o', color='steelblue', linewidth=2, markersize=7)
plt.axvline(x=3, color='tomato', linestyle='--', linewidth=1.5, label='Optimal K = 3')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Error)')
plt.title('Elbow Method — Finding the Best K')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print("📍 The elbow is at K = 3 → we'll use 3 clusters")

## Step 5 — Train K-Means with K = 3

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)

# Add the cluster label to each customer
df['Segment'] = kmeans.labels_

print("✅ Clustering done!")
print("\nCustomers per segment:")
print(df['Segment'].value_counts().sort_index().to_string())

## Step 6 — Visualise the Segments

In [ ]:
colours = ['steelblue', 'seagreen', 'tomato']
labels  = ['Segment 0', 'Segment 1', 'Segment 2']

plt.figure(figsize=(8, 5))

for seg in sorted(df['Segment'].unique()):
    subset = df[df['Segment'] == seg]
    plt.scatter(subset['Annual_Income'] / 1000, subset['Monthly_Spend'],
                color=colours[seg], label=labels[seg],
                alpha=0.6, edgecolors='white', s=50)

# Plot cluster centres
centres = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centres[:, 0] / 1000, centres[:, 1],
            color='black', marker='X', s=200, zorder=5, label='Cluster Centre')

plt.xlabel('Annual Income ($000s)')
plt.ylabel('Monthly Spending ($)')
plt.title('Customer Segments — After K-Means Clustering')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Step 7 — Understand Each Segment
What does each group actually look like?

In [ ]:
summary = df.groupby('Segment')[['Annual_Income', 'Monthly_Spend']].mean().round(0)
summary.columns = ['Avg Annual Income ($)', 'Avg Monthly Spend ($)']
summary.index = ['Segment 0', 'Segment 1', 'Segment 2']
print(summary.to_string())

print("\n💡 Segment Profiles:")
for seg in summary.index:
    inc = summary.loc[seg, 'Avg Annual Income ($)']
    spd = summary.loc[seg, 'Avg Monthly Spend ($)']
    if inc < 50000:
        profile = "🟦 Budget Shoppers    — Low income, low spending"
    elif spd > 5000:
        profile = "🟩 Big Spenders       — High income, high spending"
    else:
        profile = "🟥 Careful Savers     — High income, low spending"
    print(f"   {seg}: {profile}  |  Income: ${inc:,.0f}  |  Spend: ${spd:,.0f}/mo")

## Step 8 — Predict Segment for a New Customer 🛍️
Change the values and re-run!

In [ ]:
new_customer = pd.DataFrame({
    'Annual_Income' : [95000],   # $95,000 annual income
    'Monthly_Spend' : [7500],    # $7,500 monthly spend
})

new_scaled  = scaler.transform(new_customer)
segment     = kmeans.predict(new_scaled)[0]
seg_names   = {0: 'Budget Shoppers', 1: 'Big Spenders', 2: 'Careful Savers'}

print(f"Customer Income : ${new_customer['Annual_Income'][0]:,}")
print(f"Monthly Spend   : ${new_customer['Monthly_Spend'][0]:,}")
print(f"\n📍 Assigned to  : Segment {segment} — {seg_names.get(segment, 'Unknown')}")

---
## Summary

| | Supervised (Linear Regression) | Unsupervised (K-Means) |
|---|---|---|
| **Needs labels?** | ✅ Yes (e.g. price) | ❌ No — finds patterns itself |
| **Output** | A predicted number | A group / cluster label |
| **Use case** | Predict house price | Discover customer types |

**Business use cases for Customer Segmentation:**
- 🎯 Target each segment with personalised marketing
- 💌 Send different offers to Budget Shoppers vs Big Spenders
- 📦 Stock products based on what each segment buys